# RQ1: input-stage calibration and GQA pilot

Use Internet and **T4 ×2**. Push the new code first. Upload `research/results/rq1-inputs-v1.zip` using **Upload** in the notebook editor, or attach it as a private Kaggle Dataset. Results are ignored by Git, so pulling the repository does not fetch this ZIP.

Start with two GQA questions × 32 singleton interventions. Inspect the controls and runtime before enabling the 12-question × 576-token pilot. TextVQA generation is optional and independent; its interventions remain deferred. See `research/RQ1_INPUT_PILOT.md`.

Import this notebook into an editable draft. Default flags prevent accidental execution when Kaggle imports/runs a version.


In [ ]:
import json
import shutil
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_CALIBRATION = False  # Set True for the first two-question GQA run.
RUN_GQA_PILOT = False    # Later: True after calibration controls/timing look sensible.
RUN_TEXTVQA = False      # Optional: all 100 questions, baseline + input-only generation.
RUN_MMSTAR_CHECK = False # Later: 3 questions x 32 tokens, after GQA measurement validation.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None  # Optional exact path, e.g. /kaggle/input/my-rq1-inputs/rq1-inputs-v1.zip
CHECKPOINT_ARGS = []  # Optional: ["--model", "/kaggle/input/...", "--vision-model", "/kaggle/input/..."]

if not any([RUN_CALIBRATION, RUN_GQA_PILOT, RUN_TEXTVQA, RUN_MMSTAR_CHECK]):
    raise RuntimeError("Enable RUN_CALIBRATION first, then execute cells manually.")
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)
original_python = repo / "environments/original/.venv/bin/python"
inputs = Path("/kaggle/working/rq1-inputs-v1")
print("Repository:", repo)


## Prepare accepted inputs and reuse the original environment

This keeps the notebook kernel unchanged. The existing environment and `/tmp/apet-original-hf-cache` are reused within the same live session. Fresh sessions install/download again. Tests run before loading pretrained weights.


In [ ]:
if not (inputs / "protocol.json").exists():
    input_zip = Path(INPUT_ZIP) if INPUT_ZIP else None
    if input_zip is None:
        zip_candidates = list(Path("/kaggle/input").rglob("rq1-inputs-v1.zip"))
        zip_candidates += list(Path("/kaggle/working").glob("rq1-inputs-v1.zip"))
        extracted = []
        for path in Path("/kaggle/input").rglob("protocol.json"):
            try:
                if json.loads(path.read_text()).get("version") == "rq1-input-zero-v1":
                    if all((path.parent / name / "manifest.json").is_file() for name in ["gqa", "mmstar", "textvqa"]):
                        extracted.append(path.parent)
            except (ValueError, OSError):
                pass
        if len(zip_candidates) == 1:
            input_zip = zip_candidates[0]
        elif len(extracted) == 1:
            inputs = extracted[0]  # Kaggle may unpack a ZIP uploaded as a Dataset.
        else:
            raise RuntimeError(f"Attach rq1-inputs-v1.zip or its extracted files, and set INPUT_ZIP if needed. ZIPs: {zip_candidates}; folders: {extracted}")
    if input_zip is not None and input_zip.is_dir():
        inputs = input_zip
    elif input_zip is not None:
        inputs.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(input_zip) as z:
            for item in z.infolist():
                destination = (inputs / item.filename).resolve()
                if inputs.resolve() not in destination.parents:
                    raise ValueError("Unexpected ZIP path")
            z.extractall(inputs)
print("Accepted input directory:", inputs)
print(json.loads((inputs / "protocol.json").read_text()))
subprocess.run([
    "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
    "--output", "/kaggle/working/rq1-runtime.json",
], cwd=repo, check=True)
# Explicit import prevents Torch-dependent tests silently skipping on Kaggle.
subprocess.run([str(original_python), "-I", "-c", "import torch; print(torch.__version__)"], check=True)
for test in ["research/tests/test_rq1_input.py", "research/tests/test_rq1_native.py"]:
    subprocess.run([str(original_python), "-I", test], cwd=repo, check=True)


## First run: GQA calibration (2 questions on one image × 32 tokens)

The two questions share exact image features, native ApET reconstruction errors and sampled token indices. Controls check repeated reference scores, a clone-only no-op, cached/native forward agreement and a final reference. A failure stops the run and saves a ZIP. Download it even when a subprocess raises an error.

The runner saves reference-answer probabilities rather than asking the model to generate an answer 576 times. Positive sensitivity means the reference becomes less probable after zeroing a patch embedding; it is not an accuracy result.


In [ ]:
def run_experiment(label, manifest, options):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    folder = Path(f"/kaggle/working/{label}-{stamp}")
    command = [str(original_python), "-I", "scripts/run_rq1_input.py",
               "--manifest", str(inputs / manifest), "--output", str(folder),
               "--download-budget-gib", "20", *CHECKPOINT_ARGS, *options]
    print("Run directory:", folder, flush=True)
    log = folder.with_suffix(".log")
    with log.open("w") as stream:
        process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
            status = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if (folder / "report.json").exists():
        report = json.loads((folder / "report.json").read_text())
        print("Status:", report["status"], "wall seconds:", round(report["wall_seconds"], 1))
        for question in report.get("questions", []):
            print(question)
    print("Download:", folder.with_suffix(".zip"), "and", log)
    if status:
        raise RuntimeError(f"Run failed (exit {status}); download the report ZIP/log before restarting.")
    return folder

calibration_dir = None
if RUN_CALIBRATION:
    calibration_dir = run_experiment("rq1-gqa-calibration", "gqa/manifest.json",
                                   ["--questions", "2", "--tokens", "32"])


## Inspect calibration before expanding

Download the calibration ZIP and open `analysis/index.html` after extracting it locally. It contains the actual 336×336 CLIP canvas, native error map, sensitivity map and per-question scatter plot. Gray patches were not measured. `analysis/tokens.jsonl` preserves all 576 indices with `null` for unmeasured sensitivity.

A passed execution is a measurement check, not an established RQ1 finding. Send the calibration ZIP first. The extrapolation below includes repeated controls but excludes checkpoint loading and is only a rough estimate. Keep the full-pilot flag off until the measurement looks sensible.


In [ ]:
if calibration_dir is not None:
    report = json.loads((calibration_dir / "report.json").read_text())
    for row in report.get("questions", []):
        print(row["case_id"], "rho:", row["spearman"], "controls/status:", row["status"])
    total = sum(row["seconds"] for row in report.get("questions", []))
    if total:
        print("Rough 12 x 576 estimate (minutes):", round(total / (2 * 32) * (12 * 576) / 60, 1))
    from IPython.display import FileLink
    display(FileLink(str(calibration_dir.with_suffix(".zip"))))


## Optional next runs in the same live notebook

Each command loads the model once for its run and releases it when the subprocess exits. The environment and checkpoint downloads are reused. Enable only the run you want. Do not update repository code halfway through a running subprocess.

- GQA: 12 frozen questions on six images, including calibration pair, all 576 patches.
- TextVQA: all 100 retained questions; baseline and input-only 288 generation, native ten-answer consensus scoring. No interventions, OCR transcripts or new filtering.
- MMStar: optional three-question measurement check after GQA validation; full reviewed selection remains packaged for later expansion.


In [ ]:
if RUN_GQA_PILOT:
    gqa_dir = run_experiment("rq1-gqa-input-pilot", "gqa/manifest.json",
                             ["--questions", "12", "--tokens", "576"])


In [ ]:
if RUN_TEXTVQA:
    textvqa_dir = run_experiment("rq1-textvqa-baseline", "textvqa/manifest.json",
        ["--mode", "textvqa", "--configurations", "baseline", "input-only"])


In [ ]:
if RUN_MMSTAR_CHECK:
    mmstar_dir = run_experiment("rq1-mmstar-calibration", "mmstar/manifest.json",
                                ["--questions", "3", "--tokens", "32"])


## Download before stopping the session

In the Output panel download each `rq1-*.zip` and its `.log`. Keep the original ZIPs under `research/results/` locally. A ZIP includes predictions or token measurements, manifests, used images, source snapshots and the exact original `uv.lock`. No model weights or environment directories are bundled.

To rerun, use a new timestamped output directory. This small pilot has token checkpoints for inspection, but does not implement resume; changing controls or measurement policy should create a new run.
